# 02 — Dataset Preparation

RLDX-1 consumes datasets in the [LeRobot](https://github.com/huggingface/lerobot)
format. This notebook walks through:

1. The on-disk directory layout RLDX-1 expects.
2. What a *modality config* is and how to write one.
3. How to register a new embodiment.
4. How dataset statistics are computed and where they live.
5. How to sanity-check your dataset with the RLDX-1 dataloader.

No GPU is required for this notebook.

## 1. LeRobot dataset layout

A minimal LeRobot dataset root looks like:

```
my_dataset/
├── meta/
│   ├── info.json            # episode count, fps, feature schema
│   ├── episodes.jsonl       # one line per episode (length, index)
│   ├── tasks.jsonl          # language task strings per episode
│   └── modality.json        # RLDX-specific modality config (see §2)
├── data/
│   └── chunk-000/
│       ├── episode_000000.parquet   # per-step state & action
│       └── ...
└── videos/
    └── chunk-000/
        └── observation.images.left_view/
            ├── episode_000000.mp4
            └── ...
```

The parquet shards carry numerical state / action per step, and
the mp4 files carry per-view video. `meta/modality.json` tells
RLDX how to slice those columns into the `video` / `state` / `action`
modalities the model consumes.


## 2. Modality configs

A *modality config* is a dict of `{video, state, action, language}` →
`ModalityConfig` entries. Each entry names the keys in your parquet /
video files that RLDX should pull, and the temporal indices to pull
them at.

`rldx/configs/data/robocasa_config.py` is the canonical reference.
Let's inspect it:


In [ ]:
from rldx.configs.data.robocasa_config import robocasa_panda_omron
from pprint import pprint

print('top-level keys:', list(robocasa_panda_omron.keys()))
print()
print('video modality keys:', robocasa_panda_omron['video'].modality_keys)
print('video delta_indices:', robocasa_panda_omron['video'].delta_indices)
print()
print('state modality keys:', robocasa_panda_omron['state'].modality_keys)
print()
print('action modality keys:', robocasa_panda_omron['action'].modality_keys)
print('action horizon      :', len(robocasa_panda_omron['action'].delta_indices))


### What `delta_indices` means

- Video: `[0]` = pull only the current frame per view. A history
  (e.g. `[-2, -1, 0]`) is computed automatically by the launcher from
  `--video-length` / `--video-stride` — see `launch_train.py`. Video tokens are always present.
- State: `[0]` = current-step state only.
- Action: `list(range(16))` = predict a 16-step action chunk
  starting from the current step. This is the **action horizon**.


## 3. Writing a custom modality config

Suppose you have a single-arm dataset with one camera, 7-DoF joint
state, and a 7-DoF joint-position action chunk of length 16. The
modality config would look like the cell below. Save it as a Python
file and pass it to training with `--modality-config-path`.


In [ ]:
from rldx.data.types import (
    ActionConfig,
    ActionFormat,
    ActionRepresentation,
    ActionType,
    ModalityConfig,
)

my_modality_config = {
    'video': ModalityConfig(
        delta_indices=[0],
        modality_keys=['front_cam'],
    ),
    'state': ModalityConfig(
        delta_indices=[0],
        modality_keys=['joint_positions'],
    ),
    'action': ModalityConfig(
        delta_indices=list(range(16)),
        modality_keys=['joint_positions'],
        action_configs=[
            ActionConfig(
                rep=ActionRepresentation.ABSOLUTE,
                type=ActionType.NON_EEF,
                format=ActionFormat.DEFAULT,
            ),
        ],
    ),
    'language': ModalityConfig(
        delta_indices=[0],
        modality_keys=['annotation.human.action_tasks'],
    ),
}
print({k: type(v).__name__ for k, v in my_modality_config.items()})


## 4. Registering an embodiment

`rldx/configs/data/embodiment_configs.py` maps an `EmbodimentTag`
enum value to a modality config. `register_modality_config` adds
your dict to that mapping at import time so the training launcher
can find it via the `--embodiment-tag` CLI flag.

If your dataset is one-off and does not need an enum entry, you can
instead pass `--embodiment-tag GENERAL_EMBODIMENT` and
`--modality-config-path path/to/my_config.py`. The launcher will
import that file and expect a module-level dict whose name matches
the file (see `load_modality_config`).


In [ ]:
from rldx.configs.data.embodiment_configs import register_modality_config
from rldx.data.embodiment_tags import EmbodimentTag

# Only call this once per process — the registry is a module global.
# register_modality_config(EmbodimentTag.GENERAL_EMBODIMENT, my_modality_config)

print('available embodiment tags:', [t.value for t in EmbodimentTag][:10], '...')


## 5. Dataset statistics

Normalization statistics (per-key mean / std, or optional
percentiles) are computed from the training dataset on the first
run and then cached into the output checkpoint under
`processor/statistics.json` and `experiment_cfg/dataset_statistics.json`.

You do not have to precompute anything — `launch_train.py` will do
it automatically. If you want to reuse statistics from a prior run
(e.g. to keep a student aligned with a teacher), pass the parent
checkpoint via `--base-model-path` and the processor will load
those statistics from `processor/statistics.json`.


## 6. Sanity-check the dataloader

Before committing to a long training run, load one sample directly
through the RLDX-1 dataset factory and print its shapes. Replace
`DATA_PATH` with your dataset root.


In [ ]:
# Expected to fail cleanly until DATA_PATH points at a real dataset.
DATA_PATH = '/path/to/your/lerobot_dataset'  # TODO: point me at real data

# `LeRobotEpisodeLoader` is the lowest-level class that parses a
# LeRobot directory and honours your modality config. It returns
# one episode per index (not one step), which is the quickest way
# to confirm that videos decode and the modality keys resolve.
from rldx.data.dataset.lerobot_episode_loader import LeRobotEpisodeLoader

try:
    loader = LeRobotEpisodeLoader(
        dataset_path=DATA_PATH,
        modality_configs=my_modality_config,
    )
    print('num episodes:', len(loader))
    ep = loader[0]
    print('episode 0 type :', type(ep).__name__)
except FileNotFoundError as e:
    print('dataset not found — replace DATA_PATH above.')
    print(e)


## 7. Next

- Fine-tune the pre-trained model on the dataset you just validated →
  [`03_finetuning.ipynb`](03_finetuning.ipynb)
